# Exact reproduction — Sensitivity analysis of the SEIR–SEI dengue model

**Source:** Ganga Ram Phaijoo & Dil Bahadur Gurung, *Sensitivity Analysis of
SEIR–SEI Model of Dengue Disease*, GAMS Journal of Mathematics and Mathematical
Biosciences **6(a)**, 41–50, December 2018.

Entirely closed-form: no code or data to obtain, so a reproduction either matches
to six decimals or something is wrong.

## Targets

| # | Target | Source |
|---|---|---|
| T1 | `R0 = sqrt(α δ ν_h ν_v / (β γ ε (ε + ν_v)))` equals `ρ(F V⁻¹)` | §3 |
| T2 | The nine sensitivity indices of Table 1 | §3, Table 1 |
| T3 | Closed-form indices agree with finite differences | independent check |

## Model, in the paper's notation

```
α = b·β_h·π_v / (N_h·μ_v)     β = ν_h + μ_h
γ = γ_h + μ_h                  δ = b·β_v          ε = μ_v
```

In [ ]:
import numpy as np
import pandas as pd

CHECKS = []


def record(check_id, claim, expected, observed, passed):
    CHECKS.append((check_id, claim, expected, observed, bool(passed)))
    print(f"{check_id}  {'PASS' if passed else 'FAIL'}  {claim}")


# Parameter values listed for the simulations in Section 4.
P = dict(N_h=5_071_126, b=0.5, beta_h=0.75, beta_v=0.375, mu_h=0.000046,
         mu_v=0.25, nu_h=0.1667, nu_v=0.1428, gamma_h=0.328833, pi_v=2_500_000)

# The Sensitivity Indices column of Table 1, as printed.
TABLE1 = {"pi_v": +0.5, "b": +1.0, "beta_v": +0.5, "mu_v": -1.31823,
          "nu_v": +0.318228, "beta_h": +0.5, "gamma_h": -0.49993,
          "mu_h": -0.000208, "nu_h": +0.000138}


def composites(p):
    return dict(alpha=p["b"] * p["beta_h"] * p["pi_v"] / (p["N_h"] * p["mu_v"]),
                beta=p["nu_h"] + p["mu_h"],
                gamma=p["gamma_h"] + p["mu_h"],
                delta=p["b"] * p["beta_v"],
                epsilon=p["mu_v"])


for k, v in composites(P).items():
    print(f"{k:8s} = {v:.8f}")

## T1 — `R0` two independent ways

In [ ]:
def r0_closed(p):
    c = composites(p)
    num = c["alpha"] * c["delta"] * p["nu_h"] * p["nu_v"]
    den = c["beta"] * c["gamma"] * c["epsilon"] * (c["epsilon"] + p["nu_v"])
    return float(np.sqrt(num / den))


def r0_spectral(p):
    c = composites(p)
    F = np.array([[0, 0, 0, c["alpha"]], [0, 0, c["delta"], 0], [0, 0, 0, 0], [0, 0, 0, 0]],
                 dtype=float)
    V = np.array([[c["beta"], 0, 0, 0],
                  [0, c["epsilon"] + p["nu_v"], 0, 0],
                  [-p["nu_h"], 0, c["gamma"], 0],
                  [0, -p["nu_v"], 0, c["epsilon"]]], dtype=float)
    return float(max(abs(np.linalg.eigvals(F @ np.linalg.inv(V)))))


a, b = r0_closed(P), r0_spectral(P)
print(f"R0 (closed form) = {a!r}")
print(f"R0 (rho(F V^-1)) = {b!r}")
record("T1", "closed-form R0 equals the next-generation spectral radius",
       "identical to floating-point precision", f"diff {abs(a - b):.2e}", abs(a - b) < 1e-12)

## T2 and T3 — Table 1's nine sensitivity indices

In [ ]:
def indices_closed(p):
    mh, mv, nh, nv, gh = p["mu_h"], p["mu_v"], p["nu_h"], p["nu_v"], p["gamma_h"]
    return {"pi_v": 0.5, "b": 1.0, "beta_v": 0.5, "beta_h": 0.5,
            "mu_v": -0.5 * (2.0 + mv / (mv + nv)),
            "nu_v": 0.5 * mv / (mv + nv),
            "gamma_h": -0.5 * gh / (gh + mh),
            "nu_h": 0.5 * mh / (nh + mh),
            "mu_h": -0.5 * (mh / (nh + mh) + mh / (gh + mh))}


def indices_numeric(p, rel_step=1e-6):
    base = r0_closed(p)
    out = {}
    for name in TABLE1:
        q = p[name]
        h = q * rel_step
        up, dn = dict(p), dict(p)
        up[name], dn[name] = q + h, q - h
        out[name] = (r0_closed(up) - r0_closed(dn)) / (2 * h) * q / base
    return out


closed, numeric = indices_closed(P), indices_numeric(P)
table = pd.DataFrame([{"parameter": k, "paper Table 1": v,
                       "reproduced (closed form)": closed[k],
                       "reproduced (finite diff)": numeric[k],
                       "abs error vs paper": abs(closed[k] - v)} for k, v in TABLE1.items()])
pd.set_option("display.float_format", lambda x: f"{x: .6f}")
print(table.to_string(index=False))

worst = table["abs error vs paper"].max()
record("T2", "all nine Table 1 sensitivity indices reproduce",
       "agreement to the paper's printed precision (<= 5e-6)",
       f"largest absolute error {worst:.2e}", worst <= 5e-6)
record("T3", "closed-form indices agree with central finite differences", "~1e-6",
       "see table",
       np.allclose(table["reproduced (closed form)"], table["reproduced (finite diff)"],
                   atol=1e-6))

## Verdict

In [ ]:
from pathlib import Path

verdict = pd.DataFrame(CHECKS, columns=["check", "claim", "expected", "observed", "passed"])
print(f"{int(verdict.passed.sum())}/{len(verdict)} checks passed")

out = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
verdict.to_csv(out / "seir_sei_checks.csv", index=False)
table.to_csv(out / "seir_sei_table1_indices.csv", index=False)
print("wrote", out / "seir_sei_checks.csv", "and", out / "seir_sei_table1_indices.csv")
table

## Note on defects found

This reproduction succeeds, but the paper has three internal inconsistencies,
documented with evidence in `crosscheck/FINDINGS.md` (F4.2–F4.4):

1. Table 1's "Baseline Values" column does not produce Table 1's own indices —
   the indices follow the §4 simulation values instead.
2. Those §4 values give `R0 ≈ 0.78 < 1`, a disease-free regime, which sits badly
   with Figures 2–3.
3. The printed endemic equilibrium is not a fixed point in its vector
   components: as printed `e_v*/i_v* = ε`, while `di_v/dt = 0` requires `ε/ν_v`.

None of the three affect the results reproduced above.